# Sophisticated Controllable Agent：把 RAG 做成“可控的代理循环”

原仓库这一节链接到一个独立项目（Controllable-RAG-Agent）。这里我们用 **LangChain 的最新推荐写法**，复现它的学习目标：

- 把“检索”做成一个 tool，让模型在对话中**显式决定**是否/何时检索
- 用一个**可控的执行循环**（最大步数、每次最多检索多少文档、是否强制先检索）来约束行为
- 最终输出一个**结构化答案**（便于你把它接到后续评测/流水线）

In [9]:
import os
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import pypdf

from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 准备一个最小知识库（PDF → chunks → 向量库）

为了把 agent 的“可控循环”讲清楚，我们先准备一个最小的本地知识库：

- 下载 `Understanding_Climate_Change.pdf`
- 切 chunk
- 建 FAISS 向量库

后面 `retrieve(...)` tool 就是从这个向量库里取证据。

In [11]:
def download_file(url: str, dst: Path, *, timeout_s: float = 60.0) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst

    r = requests.get(url, timeout=timeout_s, allow_redirects=True)
    r.raise_for_status()
    dst.write_bytes(r.content)
    return dst


PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = download_file(PDF_URL, DATA_DIR / "Understanding_Climate_Change.pdf")

str(PDF_PATH)

'../data/Understanding_Climate_Change.pdf'

In [12]:
def load_pdf_pages(file_path: Path, *, max_pages: int | None = None) -> list[Document]:
    reader = pypdf.PdfReader(str(file_path))
    pages = reader.pages
    if max_pages is not None:
        pages = pages[:max_pages]

    docs: list[Document] = []
    for i, page in enumerate(pages):
        text = page.extract_text() or ""
        docs.append(Document(page_content=text, metadata={"source": str(file_path), "page": i}))
    return docs


MAX_PAGES = 8
page_docs = load_pdf_pages(PDF_PATH, max_pages=MAX_PAGES)

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunk_docs = splitter.split_documents(page_docs)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunk_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

len(chunk_docs)

22

## Step 1：把“检索”做成一个 tool（模型可选择调用）

LangChain 的 tool calling 是一个循环：

- 模型先返回“我要调用哪个 tool + 参数”
- 你执行 tool，把结果作为 `ToolMessage` 回传给模型
- 模型基于 tool 结果再继续推理或直接回答

我们这里用一个简单的 `retrieve(query)` tool，把检索到的 chunks 格式化成带编号的证据块（方便后续引用）。

In [13]:
def format_docs(docs: list[Document]) -> str:
    blocks: list[str] = []
    for i, d in enumerate(docs, start=1):
        meta = d.metadata or {}
        src = meta.get("source", "")
        page = meta.get("page", "")
        header = f"[D{i}] source={src} page={page}".strip()
        blocks.append(f"{header}\n{d.page_content}")
    return "\n\n".join(blocks)


@tool
def retrieve(query: str) -> str:
    """从本地向量库检索与 query 相关的证据片段。"""
    docs = retriever.invoke(query)
    return format_docs(docs)

## Step 2：定义“可控的代理循环”

所谓“controllable”，本质就是把关键策略显式化成几个旋钮：

- `MAX_STEPS`：最多让模型进行多少轮 tool 调用（防止无限循环）
- `FORCE_RETRIEVE_FIRST`：是否强制第一步必须检索
- `tool_choice`：是否允许模型自由选择 tool（或强制某个 tool）

同时，我们把最终回答做成结构化输出：`answer + citations`。这样你后面要做评测/落盘会很舒服。

In [14]:
class FinalAnswer(BaseModel):
    answer: str = Field(description="最终回答")
    citations: list[str] = Field(description="使用到的证据编号列表，例如 ['D1', 'D3']")


SYSTEM_PROMPT = """你是一个严格的 RAG agent。

规则：
- 当你需要事实依据时，先调用 retrieve 工具获取证据。
- 最终回答必须基于证据；如果证据不足，继续检索，而不是编造。
- 在 answer 中用 [D1] / [D2] ... 的形式做引用。
- citations 字段要列出你实际使用到的证据编号（例如 D1、D3）。
"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 这个模型负责：产生 tool_calls（也就是“要不要检索/检索什么”）
agent_llm = llm.bind_tools([retrieve])

# 这个模型负责：把最终答案输出成结构化格式（Pydantic）
final_llm = llm.with_structured_output(FinalAnswer, method="json_schema", strict=True)

MAX_STEPS = 4
FORCE_RETRIEVE_FIRST = True

In [15]:
def run_controllable_agent(question: str) -> dict[str, Any]:
    # messages 里同时会放：用户/模型消息 + tool 执行结果
    messages: list[Any] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # 可控策略：强制第一步必须检索
    # 关键点：OpenAI 要求 role='tool' 的消息必须紧跟在一个包含 tool_calls 的 assistant 消息之后。
    # 所以“强制首检索”的正确做法是：强制模型先产生 retrieve 的 tool_call，再回传 ToolMessage。
    steps_left = MAX_STEPS
    if FORCE_RETRIEVE_FIRST:
        force_llm = llm.bind_tools([retrieve], tool_choice="retrieve")
        ai_msg = force_llm.invoke(messages)
        messages.append(ai_msg)

        for tool_call in ai_msg.tool_calls or []:
            if tool_call.get("name") == "retrieve":
                evidence = retrieve.invoke(tool_call["args"])
                messages.append(
                    ToolMessage(
                        content=evidence,
                        tool_call_id=tool_call["id"],
                    )
                )

        steps_left = max(0, MAX_STEPS - 1)

    # 代理循环：最多 steps_left 轮
    for _ in range(steps_left):
        ai_msg = agent_llm.invoke(messages)
        messages.append(ai_msg)

        # 没有 tool_calls 代表模型准备直接回答（或至少不再检索）
        if not getattr(ai_msg, "tool_calls", None):
            break

        # 执行 tool_calls，把结果回传
        for tool_call in ai_msg.tool_calls:
            if tool_call.get("name") == "retrieve":
                evidence = retrieve.invoke(tool_call["args"])
                messages.append(
                    ToolMessage(
                        content=evidence,
                        tool_call_id=tool_call["id"],
                    )
                )

    # 最后把对话历史交给 final_llm，让它输出结构化答案
    final = final_llm.invoke(messages)
    return {
        "question": question,
        "final": final,
        "messages": messages,
    }

In [ ]:
question = "What is the primary cause of recent climate change?"
res = run_controllable_agent(question)

{
    "answer": res["final"].answer,
    "citations": res["final"].citations,
}

{'answer': 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere, primarily due to human activities. These activities include the burning of fossil fuels (such as coal, oil, and natural gas) and deforestation, which release significant amounts of carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). These greenhouse gases trap heat from the sun, leading to a "greenhouse effect" that warms the planet. The Intergovernmental Panel on Climate Change (IPCC) has documented that these human-induced emissions are the main drivers of the rapid changes observed in global temperatures, sea levels, and extreme weather events.',
 'citations': ['D1', 'D2']}

## 你可以怎么“控制”这个 agent？

上面这个实现有意把控制点暴露成变量，你可以按自己的场景改：

- `FORCE_RETRIEVE_FIRST=False`：让模型自己决定是否检索（更像“自适应 agent”）
- `MAX_STEPS`：调大/调小，控制工具调用上限
- `retriever.search_kwargs['k']`：控制每次检索拿多少证据

本质上：你把 agent 的自由度收敛到你能接受的范围内（成本、稳定性、可追溯性）。
